# Exploratory Data Analysis: Health Risk Prediction System

**Academic Context:** B.Tech CSE (AI & ML) Major Project  
**Dataset:** UCI Heart Disease (Cleveland Database, N=303)  
**Objective:** Understand clinical feature distributions, missingness patterns, target balance, and correlations before model training.

> **Ethical & Clinical Disclaimer:**  
> This analysis is strictly for educational and research exploration. Features identify statistical associations within a historical sample and must never be interpreted as establishing direct medical causation or diagnostic certainty.

## 1. Setup and Environment

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
PROJECT_ROOT = Path('.').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing import load_dataset, prepare_features_and_target, COLUMN_NAMES

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Environment and libraries initialized.')

## 2. Dataset Loading and Schema Inspection

In [ ]:
data_path = os.path.join(PROJECT_ROOT, 'data', 'raw', 'heart_disease_raw.csv')
df = load_dataset(data_path)
print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')
df.info()

### Findings:
- 303 patient records across 14 columns.
- Missing values represented as `?` in the raw data were parsed as `NaN`.
- Let's quantify missingness across attributes.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage (%)': missing_pct})
missing_df[missing_df['Missing Count'] > 0]

## 3. Target Distribution and Binarization

In the original Cleveland dataset:
- `num = 0`: Absence of coronary artery disease (<50% diameter stenosis).
- `num = 1, 2, 3, 4`: Presence of coronary artery disease (>=50% diameter stenosis).

For binary classification, we map `num > 0` to 1.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Multiclass target distribution
sns.countplot(data=df, x='num', ax=axes[0], palette='Blues_d')
axes[0].set_title('Original Target (num: 0 to 4)', fontsize=11)
axes[0].set_xlabel('Angiographic Disease Severity Stage')
axes[0].set_ylabel('Patient Count')

# Binarized target distribution
df['target'] = (df['num'] > 0).astype(int)
sns.countplot(data=df, x='target', ax=axes[1], palette=['#3b82f6', '#ef4444'])
axes[1].set_title('Binarized Target (0: No Disease, 1: Heart Disease)', fontsize=11)
axes[1].set_xticklabels(['No Disease (0)', 'Disease (1)'])
axes[1].set_xlabel('Classification Target')
axes[1].set_ylabel('Patient Count')

plt.tight_layout()
plt.show()

print('Class proportions:')
print(df['target'].value_counts(normalize=True))

## 4. Continuous Feature Analysis

We inspect continuous biomarkers: Age, Resting Blood Pressure (`trestbps`), Serum Cholesterol (`chol`), Maximum Heart Rate (`thalach`), and ST Depression (`oldpeak`).

In [ ]:
num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
fig, axes = plt.subplots(len(num_cols), 2, figsize=(12, 16))

for i, col in enumerate(num_cols):
    sns.histplot(data=df, x=col, hue='target', kde=True, ax=axes[i, 0], palette=['#3b82f6', '#ef4444'])
    axes[i, 0].set_title(f'{col} Distribution by Health Status', fontsize=10)
    
    sns.boxplot(data=df, x='target', y=col, ax=axes[i, 1], palette=['#3b82f6', '#ef4444'])
    axes[i, 1].set_xticklabels(['No Disease', 'Disease'])
    axes[i, 1].set_title(f'{col} Boxplot by Target', fontsize=10)

plt.tight_layout()
plt.show()

### Clinical Observations:
1. **Age:** Patients with heart disease tend to be older on average (~56.6 vs ~52.5 years).
2. **Thalach (Max Heart Rate):** Patients with heart disease demonstrate markedly lower maximum exercise heart rate.
3. **Oldpeak (ST Depression):** Pronounced exercise-induced ST depression is significantly associated with heart disease presence.

## 5. Categorical Feature Distributions by Target

In [ ]:
cat_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
fig, axes = plt.subplots(4, 2, figsize=(13, 16))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    sns.countplot(data=df, x=col, hue='target', ax=axes[i], palette=['#3b82f6', '#ef4444'])
    axes[i].set_title(f'{col} vs Target', fontsize=10)
    axes[i].legend(['No Disease', 'Heart Disease'], loc='upper right')

plt.tight_layout()
plt.show()

## 6. Correlation Analysis

In [ ]:
numeric_df = df.copy()
for c in numeric_df.columns:
    numeric_df[c] = pd.to_numeric(numeric_df[c], errors='coerce')

plt.figure(figsize=(10, 8))
corr = numeric_df.corr()
sns.heatmap(corr, cmap='vlag', annot=True, fmt='.2f', vmin=-1, vmax=1, cbar_kws={'label': 'Pearson Correlation'})
plt.title('Feature Correlation Matrix (Cleveland Database)', fontsize=12, pad=12)
plt.tight_layout()
plt.show()

## 7. Key Takeaways & Transition to Model Pipeline

1. **Balanced Distribution:** Target class balance (54.1% No Disease vs 45.9% Disease) allows for balanced classification without synthetic oversampling.
2. **Leakage Prevention:** To avoid data leakage, all imputation and scaling must be encapsulated in Scikit-learn pipelines fitted strictly on training data.
3. **Clinical Priority (Recall):** Evaluation must emphasize sensitivity/recall to minimize False Negatives.